In [1]:
import os
import shutil
import yaml
from pathlib import Path

# ===============================
# CONFIGURATION
# ===============================
RAABIN_PATH = "/Users/mahizhan/Downloads/raabin.v1i.yolov11"
NEW_PATH = "/Users/mahizhan/Downloads/wbc.v1i.yolov11"   # NEW DATASET
OUTPUT_PATH = "/Users/mahizhan/Downloads/Combined_2"

UNIFIED_CLASSES = [
    "basophil",
    "eosinophil",
    "lymphocyte",
    "monocyte",
    "neutrophil"
]

RAABIN_NAME_MAP = {
    "baso": "basophil",
    "eosi": "eosinophil",
    "lymp": "lymphocyte",
    "mono": "monocyte",
    "neut": "neutrophil"
}

# ===============================
# FUNCTIONS
# ===============================

def load_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def create_output_dirs():
    for split in ["train", "val", "test"]:
        os.makedirs(f"{OUTPUT_PATH}/{split}/images", exist_ok=True)
        os.makedirs(f"{OUTPUT_PATH}/{split}/labels", exist_ok=True)

def create_class_mapping(source_names, dataset_type):
    mapping = {}

    for idx, name in enumerate(source_names):

        if dataset_type == "raabin":
            norm_name = RAABIN_NAME_MAP[name.lower()]
        else:
            norm_name = name.lower()

        if norm_name not in UNIFIED_CLASSES:
            raise ValueError(f"{norm_name} not in unified class list")

        mapping[idx] = UNIFIED_CLASSES.index(norm_name)

    return mapping

def process_split(dataset_path, prefix, split_name, class_mapping):

    split_out = "val" if split_name == "valid" else split_name

    img_src = Path(dataset_path) / split_name / "images"
    lbl_src = Path(dataset_path) / split_name / "labels"

    img_dst = Path(OUTPUT_PATH) / split_out / "images"
    lbl_dst = Path(OUTPUT_PATH) / split_out / "labels"

    if not img_src.exists():
        return

    for img_file in img_src.glob("*.*"):

        new_img_name = f"{prefix}_{img_file.name}"
        shutil.copy(img_file, img_dst / new_img_name)

        label_file = lbl_src / (img_file.stem + ".txt")

        if label_file.exists():
            new_label_name = f"{prefix}_{img_file.stem}.txt"
            new_label_path = lbl_dst / new_label_name

            with open(label_file, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue

                old_class = int(parts[0])
                new_class = class_mapping[old_class]
                parts[0] = str(new_class)

                new_lines.append(" ".join(parts) + "\n")

            with open(new_label_path, "w") as f:
                f.writelines(new_lines)

def write_final_yaml():
    data = {
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 5,
        "names": UNIFIED_CLASSES
    }

    with open(f"{OUTPUT_PATH}/data.yaml", "w") as f:
        yaml.dump(data, f)

# ===============================
# MAIN
# ===============================

print("Loading YAML files...")

raabin_yaml = load_yaml(f"{RAABIN_PATH}/data.yaml")
new_yaml = load_yaml(f"{NEW_PATH}/data.yaml")

print("Creating class mappings...")

raabin_mapping = create_class_mapping(raabin_yaml["names"], "raabin")
new_mapping = create_class_mapping(new_yaml["names"], "new")

print("Creating output directories...")
create_output_dirs()

print("Merging RAABIN...")
for split in ["train", "valid", "test"]:
    process_split(RAABIN_PATH, "raabin", split, raabin_mapping)

print("Merging NEW dataset...")
for split in ["train", "valid", "test"]:
    process_split(NEW_PATH, "new", split, new_mapping)

print("Writing final data.yaml...")
write_final_yaml()

print("✅ DONE! RAABIN + NEW merged successfully.")


Loading YAML files...
Creating class mappings...
Creating output directories...
Merging RAABIN...
Merging NEW dataset...
Writing final data.yaml...
✅ DONE! RAABIN + NEW merged successfully.


In [2]:
import os
import shutil
import yaml
from pathlib import Path

# ===============================
# CONFIGURATION
# ===============================
RAABIN_PATH = "/Users/mahizhan/Downloads/raabin.v1i.yolov11"
NEW_PATH = "/Users/mahizhan/Downloads/wbc.v1i.yolov11"
OUTPUT_PATH = "/Users/mahizhan/Downloads/Combined_L"

UNIFIED_CLASSES = [
    "basophil",
    "eosinophil",
    "lymphocyte",
    "monocyte",
    "neutrophil"
]

LYMPHO_INDEX = UNIFIED_CLASSES.index("lymphocyte")

RAABIN_NAME_MAP = {
    "baso": "basophil",
    "eosi": "eosinophil",
    "lymp": "lymphocyte",
    "mono": "monocyte",
    "neut": "neutrophil"
}

# ===============================
# FUNCTIONS
# ===============================

def load_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def create_output_dirs():
    for split in ["train", "val", "test"]:
        os.makedirs(f"{OUTPUT_PATH}/{split}/images", exist_ok=True)
        os.makedirs(f"{OUTPUT_PATH}/{split}/labels", exist_ok=True)

def create_class_mapping(source_names, dataset_type):
    mapping = {}

    for idx, name in enumerate(source_names):

        if dataset_type == "raabin":
            norm_name = RAABIN_NAME_MAP[name.lower()]
        else:
            norm_name = name.lower()

        if norm_name not in UNIFIED_CLASSES:
            continue

        mapping[idx] = UNIFIED_CLASSES.index(norm_name)

    return mapping

def process_split(dataset_path, prefix, split_name, class_mapping, only_lymph=False):

    split_out = "val" if split_name == "valid" else split_name

    img_src = Path(dataset_path) / split_name / "images"
    lbl_src = Path(dataset_path) / split_name / "labels"

    img_dst = Path(OUTPUT_PATH) / split_out / "images"
    lbl_dst = Path(OUTPUT_PATH) / split_out / "labels"

    if not img_src.exists():
        return

    for img_file in img_src.glob("*.*"):

        label_file = lbl_src / (img_file.stem + ".txt")
        if not label_file.exists():
            continue

        with open(label_file, "r") as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue

            old_class = int(parts[0])

            if old_class not in class_mapping:
                continue

            new_class = class_mapping[old_class]

            # 🔥 FILTER: only keep lymphocytes if required
            if only_lymph and new_class != LYMPHO_INDEX:
                continue

            parts[0] = str(new_class)
            new_lines.append(" ".join(parts) + "\n")

        # If no valid labels remain, skip this image
        if not new_lines:
            continue

        # Copy image
        new_img_name = f"{prefix}_{img_file.name}"
        shutil.copy(img_file, img_dst / new_img_name)

        # Save filtered labels
        new_label_name = f"{prefix}_{img_file.stem}.txt"
        with open(lbl_dst / new_label_name, "w") as f:
            f.writelines(new_lines)

def write_final_yaml():
    data = {
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 5,
        "names": UNIFIED_CLASSES
    }

    with open(f"{OUTPUT_PATH}/data.yaml", "w") as f:
        yaml.dump(data, f)

# ===============================
# MAIN
# ===============================

print("Loading YAML files...")

raabin_yaml = load_yaml(f"{RAABIN_PATH}/data.yaml")
new_yaml = load_yaml(f"{NEW_PATH}/data.yaml")

raabin_mapping = create_class_mapping(raabin_yaml["names"], "raabin")
new_mapping = create_class_mapping(new_yaml["names"], "new")

create_output_dirs()

print("Merging RAABIN (all classes)...")
for split in ["train", "valid", "test"]:
    process_split(RAABIN_PATH, "raabin", split, raabin_mapping, only_lymph=False)

print("Merging NEW (only lymphocytes)...")
for split in ["train", "valid", "test"]:
    process_split(NEW_PATH, "new", split, new_mapping, only_lymph=True)

write_final_yaml()

print("✅ DONE! RAABIN + NEW (lymphocytes only) merged.")

Loading YAML files...
Merging RAABIN (all classes)...
Merging NEW (only lymphocytes)...
✅ DONE! RAABIN + NEW (lymphocytes only) merged.
